# Magnetic Field of Earth

### imports

In [9]:
!pip3 install pyIGRF
!pip3 install numpy

    sqlalchemy (>=1.4.1<2.0.0)
               ~~~~~~~~^

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
    sqlalchemy (>=1.4.1<2.0.0)
               ~~~~~~~~^

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip


In [11]:
import pyIGRF
import numpy as np
import math

In [7]:
def earth_magnetic_field(lat, lon, alt, year):
    Bx, By, Bz, Bh, Bt, Dec, Inc = pyIGRF.igrf_value(lat, lon, alt, year)
    print(f"North Component (X): {Bx} nT")
    print(f"East Component (Y): {By} nT")
    print(f"Down Component (Z): {Bz} nT")
    print(f"Total Field (Bt): {Bt} nT")
    print(f"Horizontal Intensity (Bh): {Bh} nT")
    print(f"Declination (D): {Dec}°")
    print(f"Inclination (I): {Inc}°")

# Converting Coordinates from North East Down to Latitude, Longitude, Altitude

In [22]:
# WGS84 ellipsoid parameters
a = 6378137.0  # semi-major axis (meters)
f = 1 / 298.257223563  # flattening
e = math.sqrt(2 * f - f ** 2)  # eccentricity
e2 = e**2

def lla_to_ecef(lat, lon, h):
    """Convert LLA (Latitude, Longitude, Altitude) to ECEF (X, Y, Z)."""
    lat_rad = math.radians(lat)
    lon_rad = math.radians(lon)
    
    N = a / math.sqrt(1 - e2 * math.sin(lat_rad) ** 2)
    
    X = (N + h) * math.cos(lat_rad) * math.cos(lon_rad)
    Y = (N + h) * math.cos(lat_rad) * math.sin(lon_rad)
    Z = (N * (1 - e2) + h) * math.sin(lat_rad)
    
    return X, Y, Z

def ned_to_ecef(n, e, d, lat_ref, lon_ref, h_ref):
    """Convert NED (North, East, Down) to ECEF (X, Y, Z)."""
    # Get ECEF coordinates of the reference point
    Xr, Yr, Zr = lla_to_ecef(lat_ref, lon_ref, h_ref)
    
    lat_ref_rad = math.radians(lat_ref)
    lon_ref_rad = math.radians(lon_ref)
    
    # Rotation matrix from NED to ECEF
    R = np.array([
        [-math.sin(lat_ref_rad) * math.cos(lon_ref_rad), -math.sin(lon_ref_rad), -math.cos(lat_ref_rad) * math.cos(lon_ref_rad)],
        [-math.sin(lat_ref_rad) * math.sin(lon_ref_rad), math.cos(lon_ref_rad), -math.cos(lat_ref_rad) * math.sin(lon_ref_rad)],
        [math.cos(lat_ref_rad), 0, -math.sin(lat_ref_rad)]
    ])
    
    # Convert NED to ECEF
    ned_vector = np.array([n, e, d])
    ecef_vector = R.T @ ned_vector  # Apply rotation
    
    # New ECEF position
    X = Xr + ecef_vector[0]
    Y = Yr + ecef_vector[1]
    Z = Zr + ecef_vector[2]
    
    return X, Y, Z

def ecef_to_geodetic(X, Y, Z):
    """Convert ECEF (X, Y, Z) to LLA (Latitude, Longitude, Altitude)."""
    lon = math.atan2(Y, X)
    p = math.sqrt(X ** 2 + Y ** 2)
    lat = math.atan2(Z, p * (1 - f))  # initial guess
    
    lat_prev = 0
    while abs(lat - lat_prev) > 1e-10:  
        lat_prev = lat
        sin_lat = math.sin(lat)
        N = a / math.sqrt(1 - e2 * sin_lat ** 2)
        lat = math.atan2(Z + e2 * N * sin_lat, p)
    
    N = a / math.sqrt(1 - e2 * math.sin(lat) ** 2)
    h = p / math.cos(lat) - N

def ned_to_lla(n, e, d, lat_ref, lon_ref, h_ref):
    """Convert NED coordinates to LLA using reference LLA."""
    X, Y, Z = ned_to_ecef(n, e, d, lat_ref, lon_ref, h_ref)
    return ecef_to_geodetic(X, Y, Z)

# Barometer

In [60]:
P_0 = 101.325  # kPa (standard atmospheric pressure at sea level)
T_0 = 288.15  # K (standard temperature at sea level)
g = 9.80665 # m/s^2
L = 0.0065 # K/m in the troposphere
R = 8.3144598 # universal gas constant
M = 0.0289644 # molar mass of Earth's air

def pressureToAltitude(p):
    return (T_0/L)*(1-(p/P_0)**((R*L)/(g*M)))

def altitudeToPressure(h):
    return P_0 * (1 - L * h / T_0) ** (g * M / (R * L))

# exact temperature
def altitudeTemperatureToPressure(h, t):
    return P_0*math.exp(-g*M*h/(R*t))


94.67756194232625
